In [4]:
!pip install gridstatus

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.0/528.0 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.4/159.4 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 6

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, Dataset
import gridstatus

In [6]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

class RealTimeGridDataset(Dataset):
    def __init__(self, iso_name: str = 'CAISO', days_history: int = 30, history_size: int = 168, horizons: list = [24, 48, 72]):
        print(f"Fetching last {days_history} days of LIVE data from {iso_name}...")

        if iso_name == 'CAISO':
            iso = gridstatus.CAISO()
        elif iso_name == 'ERCOT':
            iso = gridstatus.ERCOT()
        elif iso_name == 'PJM':
            iso = gridstatus.PJM()
        else:
            raise ValueError("Unsupported ISO.")

        end_date = pd.Timestamp.now(tz='UTC')
        start_date = end_date - pd.Timedelta(days=days_history)

        raw_df = iso.get_load(start=start_date, end=end_date)

        raw_df = raw_df.set_index('Interval Start')
        raw_df.index = pd.to_datetime(raw_df.index, utc=True)

        # Fixed: using lowercase 'h' for hourly resampling to resolve the FutureWarning
        hourly_df = raw_df[['Load']].resample('h').mean().ffill()
        self.df = hourly_df

        print(f"Successfully loaded {len(self.df)} hourly data points.")

        self.data = self.df.values
        self.history_size = history_size
        self.horizons = horizons

        self.scaler = RobustScaler()
        self.scaled_data = self.scaler.fit_transform(self.data)

    def __len__(self):
        return len(self.scaled_data) - self.history_size - max(self.horizons)

    def __getitem__(self, idx):
        x = self.scaled_data[idx : idx + self.history_size]
        y_24 = self.scaled_data[idx + self.history_size + 23]
        y_48 = self.scaled_data[idx + self.history_size + 47]
        y_72 = self.scaled_data[idx + self.history_size + 71]

        return torch.tensor(x, dtype=torch.float32), {
            "24h": torch.tensor(y_24, dtype=torch.float32),
            "48h": torch.tensor(y_48, dtype=torch.float32),
            "72h": torch.tensor(y_72, dtype=torch.float32)
        }

**Model architecture**

In [7]:
class GridAttention(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return torch.sum(x * weights, dim=1)

class ResidualGridNet(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True, dropout=0.2)
        self.attention = GridAttention(hidden_dim)

        self.h24 = nn.Linear(hidden_dim * 2, input_dim)
        self.h48 = nn.Linear(hidden_dim * 2, input_dim)
        self.h72 = nn.Linear(hidden_dim * 2, input_dim)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context = self.attention(lstm_out)

        last_val = x[:, -1, :]
        res_24 = self.h24(context)
        res_48 = self.h48(context)
        res_72 = self.h72(context)

        return {
            "24h": last_val + res_24,
            "48h": last_val + res_48,
            "72h": last_val + res_72
        }

def train_step(model, batch, optimizer, criterion):
    x, targets = batch
    preds = model(x)

    loss_24 = criterion(preds["24h"], targets["24h"])
    loss_48 = criterion(preds["48h"], targets["48h"])
    loss_72 = criterion(preds["72h"], targets["72h"])

    total_loss = (loss_24 * 1.0) + (loss_48 * 0.5) + (loss_72 * 0.2)

    total_loss.backward()
    optimizer.step()
    return total_loss.item()

In [8]:
# Instantiate the live dataset (Fetching 60 days of live California Grid Data)
live_dataset = RealTimeGridDataset(iso_name='CAISO', days_history=60)

# Create the PyTorch DataLoader
dataloader = DataLoader(live_dataset, batch_size=32, shuffle=True, drop_last=True)

# Initialize models and optimizers
model = ResidualGridNet(input_dim=1, hidden_dim=128)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Quick Training Loop
print("\n--- Starting Training on Live Data ---")
epochs = 5

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for batch_idx, batch in enumerate(dataloader):
        # Zero gradients at the start of each batch
        optimizer.zero_grad()
        loss = train_step(model, batch, optimizer, criterion)
        epoch_loss += loss

    avg_loss = epoch_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{epochs}] | Weighted Multi-Horizon Loss: {avg_loss:.4f}")

print("\nPipeline execution complete! The model has been trained on the latest real-time grid conditions.")

Fetching last 60 days of LIVE data from CAISO...


  0%|          | 0/61 [00:00<?, ?it/s]2026-04-16 23:01:35 - INFO - Fetching URL: https://www.caiso.com/outlook/history/20260215/demand.csv?_=1776380495
INFO:gridstatus:Fetching URL: https://www.caiso.com/outlook/history/20260215/demand.csv?_=1776380495
  2%|▏         | 1/61 [00:00<00:41,  1.46it/s]2026-04-16 23:01:35 - INFO - Fetching URL: https://www.caiso.com/outlook/history/20260216/demand.csv?_=1776380495
INFO:gridstatus:Fetching URL: https://www.caiso.com/outlook/history/20260216/demand.csv?_=1776380495
  3%|▎         | 2/61 [00:01<00:27,  2.13it/s]2026-04-16 23:01:36 - INFO - Fetching URL: https://www.caiso.com/outlook/history/20260217/demand.csv?_=1776380496
INFO:gridstatus:Fetching URL: https://www.caiso.com/outlook/history/20260217/demand.csv?_=1776380496
  5%|▍         | 3/61 [00:01<00:24,  2.38it/s]2026-04-16 23:01:36 - INFO - Fetching URL: https://www.caiso.com/outlook/history/20260218/demand.csv?_=1776380496
INFO:gridstatus:Fetching URL: https://www.caiso.com/outlook/histo

Successfully loaded 1455 hourly data points.

--- Starting Training on Live Data ---
Epoch [1/5] | Weighted Multi-Horizon Loss: 0.4356
Epoch [2/5] | Weighted Multi-Horizon Loss: 0.4312
Epoch [3/5] | Weighted Multi-Horizon Loss: 0.4266
Epoch [4/5] | Weighted Multi-Horizon Loss: 0.4295
Epoch [5/5] | Weighted Multi-Horizon Loss: 0.4230

Pipeline execution complete! The model has been trained on the latest real-time grid conditions.
